# Lekcija 03 - Agentni dizajnerski obrasci

U ovoj lekciji istražujemo tri temeljna dizajnerska obrasca za izgradnju učinkovitih AI agenata:

1. **Jasne upute za agente** — Izrada preciznih, ulogom definirajućih upita koji usmjeravaju ponašanje agenata
2. **Strukturirani izlaz s Pydantic modelima** — Osiguravanje da agenti vraćaju predvidljive, verificirane podatke
3. **Agenti s jednom odgovornosti** — Dizajniranje fokusiranih agenata koji svaki rade jednu stvar dobro

Svaki ćemo obrazac primijeniti na scenarij **preporučitelja putničkih destinacija**, postupno gradeći sustav koji može predlagati destinacije, provjeravati dostupnost i rukovati logistikom.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Uzorak 1: Jasne upute za agenta

Najutjecajniji uzorak je također i najjednostavniji: pisanje jasnih, detaljnih uputa za vašeg agenta.

Dobre upute definiraju:
- **Tko** je agent (persona i ton)
- **Što** bi trebao raditi (odgovornosti korak po korak)
- **Kako** bi se trebao ponašati (ograničenja i stil)

U nastavku stvaramo agenta turističkog conciergea s eksplicitnim uputama koje oblikuju svaki odgovor koji daje.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Uzorak 2: Strukturirani izlaz s Pydantic modelima

Slobodni tekst je koristan za razgovor, ali sustavi niže razine trebaju strukturirane podatke.
Uparivanjem **Pydantic modela** s **funkcijom alata**, možemo:

- Definirati točan shemu za izlaz agenta
- Automatski validirati odgovore
- Pouzdano integrirati rezultate agenta u logiku aplikacije

Ključ za provođenje je prosljeđivanje `response_format` kada pokrećemo agenta. To forsira
model da vrati validirani objekt `TravelRecommendations` (dostupan na `response.value`)
umjesto slobodnog teksta. Alat `get_destination_details` također vraća tipizirani
`DestinationRecommendation`, tako da podaci ostaju strukturirani od početka do kraja.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Uzorak 3: Agent s jednom odgovornošću

Složeni zadaci imaju koristi od podjele rada na više fokusiranih agenata, svaki sa svojom jednom odgovornošću:

- **Stručnjak za destinacije** koji poznaje mjesta i dostupnost
- **Planer logistike** koji organizira letove, hotele i itinerere

Ovo odražava princip softverskog inženjerstva *separacije odgovornosti* — svaki agent je lakše testirati, održavati i poboljšavati neovisno.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sažetak

U ovoj lekciji primijenili smo tri agentička dizajnerska obrasca na scenarij preporuke putovanja:

| Obrasc | Ključna ideja | Korist |
|---|---|---|
| **Jasna uputstva** | Definirati personu, odgovornosti i ograničenja unaprijed | Dosljedno, na brend usklađeno ponašanje agenta |
| **Strukturirani izlaz** | Koristiti Pydantic modele kao format odgovora | Validirani, strojno čitljivi rezultati |
| **Jedna odgovornost** | Dati svakom agentu jedan fokusirani zadatak | Lakše za testiranje, održavanje i sastavljanje |

Ovi obrasci se prirodno nadopunjuju — možete kombinirati jasna uputstva sa strukturiranim izlazom unutar agenta s jednom odgovornosti za izgradnju robusnih, proizvodno spremnih sustava.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
